# Studio 7: Verify Your Analysis from a Clean Restart

**Topic 08 · 1 lecture**

<hr>

<center>
<div>
<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" width="300"/>
</div>
</center>


# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb08_prediction_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** all positions (verification week). Whatever kind and reach
your Contract declares, this week patrols one crossing every route must earn:
the crossing from "my code printed a number" to "my result is verified." Last
week your pipeline produced a provisional first result. This week that result
faces a **clean restart**, a rerun from a fresh state with the machine's memory
cleared, and you learn what to do when the number that comes back is not the
number you submitted.

**Design pathway:** cross-cutting. The clean-restart discipline is route-neutral;
every pathway runs it the same way. Prediction routes carry one extra branch:
a **leakage and boundary audit**, because a forecast can fail in a way no other
route can, by quietly using information from the future.

| | |
|---|---|
| **A claim this topic PERMITS** | "From a clean restart in a recorded environment, my pipeline reproduces [estimate] with its uncertainty, every claim I make traces to a specific output, and I independently rederived [two quantities] by a second method." |
| **A claim this topic does NOT permit** | "The earlier (or later) output is the right one because of when it ran," and "it reproduces, so the finding is correct." A clean run verifies the pipeline, never the design or the claim. |

**Where this sits in the course:** Week 8, the semester's single-lecture week.
This lecture develops **M7, your Clean-Restart Verified Analysis**, the second
version of your first-analysis milestone (Book Milestone 7 v2), worked on and
submitted at Friday's studio. It builds directly on M6's executable pipeline and
hands a verified result to next week's stress test.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Run a **clean restart** (restart-and-run-all) and explain what it proves
   that a same-session rerun cannot.
2. Compare two **run records** field by field and read a run-to-run diff the
   way an auditor would.
3. Classify any discrepancy between runs with the five-lane taxonomy: **data,
   environment, order, code, claim trace**, and decide when the difference is
   itself the finding.
4. Independently **rederive** two reported quantities by a second method, and
   state exactly what agreement between two methods does and does not establish.
5. Run the prediction-route **leakage audit** (timing, split, boundary), and
   audit any route's claim language against the outputs that must back it.
6. Apply all of it to your own project's **M7**: the clean-run log, the
   run-record diff, two independent rederivations, and the verified
   claim-output trace.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn: used only in the prediction-route leakage branch.
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

# Lecture 1

🗺️ **The frame (50 min):** 🧩 puzzle → 🔮 **Predict First** · 🛠️ **Run the Study** with your AI · 🔍 **Read the Evidence** · 📝 **Practice** · ⚖️ **Make a Design Choice** · 🎯 **Take It to Your Project** · 🛡️ **Defend Your Decision** · 📒 ledger row. All seven moves happen in class; 🏠-marked items and everything below the ⏸ line are optional depth.

### 🎤 SRL Lead Brief

*This lecture opens with its Student Research Lead. If this is your slot, this
brief is your backbone. If not, read along: this is how the room will run.*

**Your mission.** By minute 50 the room can say what a clean restart proves,
diagnose a changed number with one of five lane words, and everyone holds a
verification plan for their own M7.

**Run of show (Wednesday: 7 / 23 / 12 / 8).**

| Min | Section | What you do |
|---|---|---|
| 0–7 | Changed-result challenge | Present the two colleague run records below, cold. Everyone commits in writing: which number is the result, A, B, or neither yet. Then run the spoken retrieval drill. |
| 7–30 | Applied laboratory | Steer the room through the common clean run and the record diff in section 1, the colleague reconstruction in section 2, the two rederivations in section 3, and the leakage branch in section 4. Keep asking one thing: which lane does that difference point to? |
| 30–38 | Peer defense | Partners attack each other's clean-restart verdicts: a crowned run, a missing rederivation, a claim with no output behind it. |
| 38–42 | Your synthesis | Put the five lanes on the board in the room's own words; your instructor locks the accuracy. |
| 42–50 | Project transfer | Everyone records the M7 decision and a ledger entry; close on a spoken Claim Ticket. |

**Three questions that keep the room thinking.**

1. *"The number changed after the restart. Which run is right?"* Listen for:
   neither, by order alone; the difference is a finding that needs a diagnosis.
   Watch out for: "the newer run must be the corrected one," the exact reflex
   this lecture exists to break.
2. *"Same notebook, same file, same seed. What could possibly have changed?"*
   Listen for: what was in the machine's memory when the estimate ran. Watch
   out for: blaming the data first; the records show the same file.
3. *"The colleague's forecast improved after the restart. Why is that the most
   suspicious line in the dossier?"* Listen for: the new feature is settled
   after the outcome, so no real forecast could use it.

**One AI trap to watch for.** Somebody will paste both records into a chatbot
and ask "which one is right?", and get back a fluent verdict crowning one run,
usually the newer one. That is a conclusion wearing no diagnosis: no lane, no
check, no execution record. When it appears, say that a verdict without a
diagnosis is just a preference with confidence.

**Checkpoints.** By minute 7, every commitment is written. By minute 30, the
record diff is on the board and both rederived numbers agree with the record.
By minute 38, every verdict in the room has survived one peer attack or been
rewritten. By minute 42, the five lanes are on the board and locked.

**Make it yours.** The frame, the records, and the checkpoints are fixed; the
staging is yours. Restage the opener as a courtroom with A and B as rival
witnesses, or have the room vote before and after the reconstruction and read
the flips aloud. In your prep script, name one thing you are adding that this
brief does not contain.

**Prep.** Start about a week ahead. Submit your preparation script or notebook
two days ahead. The full guide is the Student Research Lead handbook in the
course materials.

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

Your instructor issues two run records from a colleague's dossier.

> **SIMULATED CASE · TRAINING MATERIAL · NOT EVIDENCE**
>
> *Record A, saved when the first analysis was submitted:* "turnout gap
> **+3.41 points**, 95% interval +1.0 to +5.8; rows in the estimate:
> **8,375**; seed 464; the notebook ran with no errors."
>
> *Record B, saved days later, after Runtime, Restart and run all, on the
> unchanged notebook:* "turnout gap **+4.27 points**; rows in the estimate:
> **7,281**; seed 464; the notebook ran with no errors."
>
> *One more line in the dossier, from the colleague's prediction-route side
> project:* "Good news: after the restart my turnout forecast jumped from
> 0.65 to 0.89 held-out, once the rerun picked up my new
> `post_election_contact` feature."

Same notebook, same file, same seed, no errors either time. Here is the
question on the table: **which number is the colleague's result, +3.41 or
+4.27?** Commit in writing: A, B, or neither yet, plus one sentence of
reasoning. Then look harder at the two records. One field besides the estimate
changed, and it names exactly where to dig. And hold that forecast line for
later: something about a score that improves on its own should bother you
before any code runs.

### 📝 Practice: retrieval drill

Before any AI opens today: from memory, in about a minute, state (1) the four
parts a finished first result carries (estimate, uncertainty, frame, boundary),
(2) the two records your pipeline shipped beside its number last week, and
(3) the label that number wears until a clean restart confirms it. Answers run
aloud; nothing written is required.

## 1. What a Clean Restart Proves

**Guiding question:** *your notebook ran and printed the number you submitted,
so what is left to verify?*

> *"I do not ask whether your notebook ran. I ask whether it runs from a cold
> start, in a recorded environment, and hands back the number you claimed.
> Until it does, you are showing me a memory, not a result."*
> — a **journal reviewer** who audits computational supplements for a living

Last week your result was labeled **provisional**, pending exactly the test
this lecture performs. Three terms carry the whole hour.

- A **clean restart** (restart-and-run-all) clears the machine's working
  memory, then runs every cell from the top, in order, with no manual fixes.
  Example: in Colab, Runtime, then Restart and run all. It is the difference
  between "this notebook ran once" and "this notebook runs."
- **Hidden state** is whatever lingers in memory from cells you ran earlier,
  edited since, or ran out of order. Example: a variable that still holds the
  unfiltered table after you added a filter cell but never reran it. Hidden
  state is why a notebook can display a number its own code no longer produces.
- A **run record** is the written snapshot that makes a run checkable: the
  environment (versions, seed), the data (file, shape, fingerprint), and the
  outputs (the estimate with its uncertainty). Example: the record you built
  last week. The auditing habit this week installs is to **demand the
  execution record**: any number, from your own notebook or from an AI
  assistant's report, is only as good as the record of the run that produced
  it, its inputs, outputs, seed, and order.

And one guard before anything runs: **if the numbers move, the pipeline is the
finding.** A changed number after a clean restart is not an embarrassment to
hide. It is a discovery about your pipeline, and diagnosing it is research.

> **A question that often comes up here:** *"I reran my notebook yesterday
> without restarting, and every number matched. Is that not already
> verification?"* It is weaker than it feels. A same-session rerun runs inside
> the same memory, so any hidden state that shaped your number the first time
> is still there, shaping it the second time. The restart clears that memory,
> which is exactly why passing it means more.

**What to expect when you run the next cell:** the shared first-analysis
pipeline from last week, rerun from this fresh session, printing its result and
the full run record it produces.

In [ ]:
# THE COMMON CLEAN RUN — the shared first-analysis pipeline, rerun from a fresh state.
foos = load_course_data("foos_etal.csv")
assert foos.shape == (8375, 5), "unexpected shape — stop and investigate before anything else"
print("✓ Loaded foos_etal.csv —", foos.shape[0], "rows,", foos.shape[1], "columns")

OUTCOME, GROUP = "marked_register_2014", "treat"
t1 = foos.loc[foos[GROUP] == 1, OUTCOME]   # canvassed
t0 = foos.loc[foos[GROUP] == 0, OUTCOME]   # control
diff = t1.mean() - t0.mean()
se = np.sqrt(t1.mean() * (1 - t1.mean()) / len(t1)
             + t0.mean() * (1 - t0.mean()) / len(t0))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se

import platform
record_restart = {
    "dataset":      "foos_etal.csv",
    "rows x cols":  f"{foos.shape[0]} x {foos.shape[1]}",
    "seed":         SEED,
    "arms":         f"{len(t1)} treated / {len(t0)} control",
    "estimate_pts": round(float(diff) * 100, 2),
    "se_pts":       round(float(se) * 100, 2),
    "ci_pts":       (round(float(ci_lo) * 100, 1), round(float(ci_hi) * 100, 1)),
}
env_lane = {
    "python":           platform.python_version(),
    "pandas":           pd.__version__,
    "data fingerprint": int(pd.util.hash_pandas_object(foos).sum()) % 10**12,
}
print("\nRUN RECORD — clean restart, today:")
for k, v in record_restart.items():
    print(f"  {k:>14} : {v}")
print("  environment lane (compare against what YOUR record captured):")
for k, v in env_lane.items():
    print(f"  {k:>16} : {v}")

**Reading the output.** The rerun lands on **+3.41 points** with a standard
error of **1.23** and a 95% interval from **+1.0 to +5.8**, on 6,104 treated
and 2,271 control rows. Those are the values to hold in your head for the next
step. The record also carries an environment lane: the software versions and a
**data fingerprint**, a single number computed from every cell of the table,
so that a changed file announces itself. The record is only half the exercise.
Verification is a comparison, and the other half is the record you are
comparing against.

### 🔮 Predict First

The next step diffs today's clean-restart record against the record submitted
with the first analysis, field by field: dataset, shape, seed, arms, estimate,
uncertainty, interval. Before running it, commit two predictions in writing.
First: will every one of those fields match exactly, yes or no? Second: name
the one kind of field that could legitimately differ between two honest runs
of the same pipeline, and the kind that must never differ silently.

### YOUR ANSWER HERE:

**Every field matches (yes / no):**

**A field that may legitimately differ, and one that must not:**

---

### 🛠️ Run the Study: diff the two run records

Run the cell below. It loads the record submitted with the shared pipeline,
lines it up against today's clean-restart record, and prints the comparison
field by field, with a verdict for each row.

**🔴 Live in class: we run this one together.**
**Before you ask:** in one sentence, write which single field of a run-record
diff you would check first, and why that one.

> 💡 **AI Prompt:** "Here are two run records from the same analysis notebook,
> one saved at submission and one saved after a clean restart: [paste the
> printed records]. For each field that differs, name which failure lane it
> points to (data, environment, order, code, or claim trace) and the one check
> that would confirm the diagnosis. Do not declare either run correct, and say
> explicitly what an all-match result does and does not establish."
>
> **After running, verify (counters *confident fabrication*: a fluent audit can
> quote fields your records do not contain and crown a run anyway):**
> - [ ] Check every field the AI names against your printed diff. A field it
>   invents, or a mismatch it reports where your table prints ✓, disqualifies
>   the audit.
> - [ ] Confirm it refused to crown a run. If it declared one record "the
>   correct one" without a diagnosis, reject that sentence and keep the lanes.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# THE RUN-TO-RUN DIFF — today's clean restart vs the record shipped at submission.
RECORD_SUBMITTED = {   # the first-analysis record, as submitted with the shared pipeline
    "dataset":      "foos_etal.csv",
    "rows x cols":  "8375 x 5",
    "seed":         464,
    "arms":         "6104 treated / 2271 control",
    "estimate_pts": 3.41,
    "se_pts":       1.23,
    "ci_pts":       (1.0, 5.8),
}

print(f"{'field':>14} | {'submitted':>24} | {'clean restart':>24} | verdict")
print("-" * 82)
mismatches = []
for k, sub_v in RECORD_SUBMITTED.items():
    new_v = record_restart[k]
    ok = new_v == sub_v
    if not ok:
        mismatches.append(k)
    print(f"{k:>14} | {str(sub_v):>24} | {str(new_v):>24} | {'✓ match' if ok else '✗ INVESTIGATE'}")

print()
if not mismatches:
    print("✓ All shared fields match: the pipeline REPRODUCES from a clean restart.")
    print("  What that establishes: the number regenerates from the files, in a fresh state.")
    print("  What it does not establish: that the design is sound or the claim is true.")
else:
    print("✗ Fields to investigate:", mismatches)

assert not mismatches, "the shared pipeline should reproduce exactly — investigate the diff"
assert abs(record_restart["estimate_pts"] - 3.41) < 0.05, "estimate drifted — investigate"
assert abs(record_restart["se_pts"] - 1.23) < 0.05, "uncertainty drifted — investigate"

**Reading the output.** Every shared field prints **✓ match**: the same file,
the same arms, the same +3.41 with the same 1.23. This pipeline passes its
clean restart, and the provisional label it has carried since last week can now
come off, for this pipeline, in this environment. Two boundaries keep the
celebration honest. First, the environment lane can legitimately differ when
you rerun in a different runtime, a library version, a Python release; a
difference there is benign *until a number moves*, and then it is the first
suspect. Second, an all-match verdict verifies the pipeline, not the claim: no
diff table can tell you whether the sentence you plan to write matches these
outputs. That check, the claim trace, stays yours.

**Section bridge:** your colleague's diff did not print ✓ down the column. Time
to find out why.

---

## 2. When the Number Moves: Diagnose Before You Crown

**Guiding question:** *the clean restart returned a different number, so which
run do you believe?*

The puzzle's two records disagree: +3.41 on 8,375 rows against +4.27 on 7,281
rows. The wrong move, and the most tempting one, is to crown a winner by
recency: "the restart is cleaner, so +4.27 is the corrected result." Resist it.
**Neither run is right because of when it ran.** The row count is the field
that talks: the estimate in Record B ran on 1,094 fewer rows, so somewhere in
that notebook, rows are being dropped in one run and kept in the other.

Here is what happened, reconstructed. While polishing the notebook after
submission, the colleague added a small "cleaning" cell that drops ward G,
whose records looked messy, *above* the estimate. In the working session, that
new cell was never run, so the estimate kept using the full table sitting in
memory: hidden state, +3.41. Restart-and-run-all executes the notebook as
written, filter first, estimate second: +4.27. The submitted number and the
shipped code had quietly parted ways.

> **A question that often comes up here:** *"So +4.27 is what the code really
> does. Doesn't that make it the right answer?"* It makes it the *reproducible*
> answer to a question nobody declared. Dropping ward G changes what is being
> estimated, and that is a design decision your Contract has to license, not a
> side effect of a cleaning cell. The finding here is the detachment itself:
> the package produced a number its own code does not produce. The fix is a
> decision (keep ward G or justify dropping it), recorded, and then one more
> clean run.

**What to expect when you run the next cell:** the colleague's notebook
reconstructed, both run orders executed side by side, and the two records'
numbers recovered exactly.

In [ ]:
# THE COLLEAGUE'S NOTEBOOK, RECONSTRUCTED — one late cleaning cell, two run orders.
def estimate_gap(df):
    """The estimate cell: turnout gap in points, plus the rows it used."""
    a = df.loc[df["treat"] == 1, "marked_register_2014"]
    b = df.loc[df["treat"] == 0, "marked_register_2014"]
    return (a.mean() - b.mean()) * 100, len(a) + len(b)

# Record A (the session): the estimate ran BEFORE the late cleaning cell ever executed,
# so it used the full table still sitting in memory.
gap_a, n_a = estimate_gap(foos)

# Record B (restart-and-run-all): the notebook runs as written — cleaning first.
cleaned = foos[foos["ward"] != "G"]        # "ward G looked messy" — added after submission
gap_b, n_b = estimate_gap(cleaned)

print(f"  Record A (session, hidden state): gap {gap_a:+.2f} points on {n_a:,} rows")
print(f"  Record B (clean restart):         gap {gap_b:+.2f} points on {n_b:,} rows")
print(f"  disagreement: {gap_b - gap_a:+.2f} points, with no error message either time")
print()
print("  Diagnosis: same file, same seed, same cells — what differed is which cells'")
print("  EFFECTS were in memory when the estimate ran. That is the ORDER lane.")
print("  Consequence: the shipped notebook no longer produces the shipped +3.41,")
print("  so the package now also carries a CLAIM-TRACE failure to repair.")

assert abs(gap_a - 3.41) < 0.05, "Record A should recover the submitted +3.41"
assert abs(gap_b - 4.27) < 0.05, "Record B should recover the clean-restart +4.27"
print("\n✓ Both records recovered. The difference was never a mystery — it was a diagnosis waiting.")

### 🔍 Read the Evidence: what each verdict licenses

Two diffs are now in front of you: the shared pipeline's all-match, and the
colleague's mismatch. In the cell below, write three things. First: one
sentence stating exactly what the all-match verdict *does* establish for the
shared pipeline, and one thing it does not. Second: the colleague verdict in
two parts, the lane word the row count points to, and the second failure the
package carries as a consequence. Third: answer the puzzle as an auditor
would: which number is the colleague's result, +3.41, +4.27, or neither yet,
and what has to happen before that answer changes.

### YOUR ANSWER HERE:

**What the all-match verdict establishes / does not establish:**

**The colleague diagnosis (lane word + the consequence failure):**

**Which number is the result, and what must happen before that changes:**

---

## 3. Rederive It Yourself: Two Quantities, a Second Method

**Guiding question:** *the pipeline agrees with itself, so how do you check
that it agrees with the data?*

A clean restart reruns the *same* code. If that code holds a quiet mistake, the
restart reproduces the mistake perfectly. The next layer of verification is
**independent rederivation**: recomputing a reported quantity by a second path
that shares as little machinery as possible with the first. Example: your
pipeline computed the turnout gap from two group means; you rederive it from
the raw counts of a cross-table, with hand arithmetic and no `.mean()` anywhere.
Your M7 record carries two of these, and this section builds both on the shared
result:

- **Quantity 1, the estimate,** rederived from the four cell counts of a
  cross-tabulation.
- **Quantity 2, the uncertainty,** rederived by a **bootstrap**: resampling the
  rows with replacement many times, recomputing the gap each time, and reading
  the spread of those recomputed gaps. Example: 2,000 resamples whose standard
  deviation stands in for the formula's standard error, no formula involved.

Before you run it, commit in writing: the formula said 1.23. Within what range
would the bootstrap spread have to land for you to call the two methods "in
agreement"? Write your tolerance down first, so the output cannot negotiate it.

### 🛠️ Run It Live: rederive the gap and its uncertainty

Run the cell below. It rederives the estimate from raw counts, rederives the
uncertainty by seeded bootstrap, and lines both up against the run record.

**🏠 Optional depth.** Run the AI prompt on your own if you want to go deeper.
**Before you ask:** in one sentence, name a failure that BOTH the pipeline and
your rederivation would share, so agreement between them could never catch it.

> 💡 **AI Prompt:** "Here are two computations of the same two quantities: a
> pipeline's difference-in-means with a formula standard error, and a
> rederivation from crosstab counts with a bootstrap spread: [paste both cells
> and their printed numbers]. Explain why agreement between the two methods
> counts as independent evidence about the computation, then name the failures
> agreement can NOT catch, such as both methods reading the same wrong file,
> and the check that would catch each one."
>
> **After running, verify (counters *illusion of completeness*: a tidy list of
> what agreement proves can quietly skip what it cannot prove):**
> - [ ] Check every number the AI quotes against your printout: the two gap
>   values and the two uncertainty values. An invented decimal disqualifies it.
> - [ ] Confirm its list includes at least one shared-input failure (same wrong
>   file, same wrong filter). If it does not, the tidy list failed the one test
>   it looked complete on.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# INDEPENDENT REDERIVATION — the same two quantities, by a second method each.
# Quantity 1: the estimate, from the crosstab's four counts (no .mean() anywhere).
ct = pd.crosstab(foos["treat"], foos["marked_register_2014"])
print("Cross-table (rows = treat, cols = voted):")
print(ct.to_string())
p1 = ct.loc[1, 1] / (ct.loc[1, 0] + ct.loc[1, 1])
p0 = ct.loc[0, 1] / (ct.loc[0, 0] + ct.loc[0, 1])
rederived_gap = (p1 - p0) * 100
print(f"\n  rederived gap from counts: {rederived_gap:+.2f} points"
      f"   (pipeline said {record_restart['estimate_pts']:+.2f})")

# Quantity 2: the uncertainty, by seeded bootstrap instead of the formula.
boot_rng = np.random.default_rng(SEED)
treat_arr = foos["treat"].to_numpy()
vote_arr = foos["marked_register_2014"].to_numpy()
boots = np.empty(2000)
for i in range(2000):
    idx = boot_rng.integers(0, len(foos), len(foos))
    t_s, v_s = treat_arr[idx], vote_arr[idx]
    boots[i] = v_s[t_s == 1].mean() - v_s[t_s == 0].mean()
boot_se = boots.std() * 100
print(f"  bootstrap spread of the gap:  {boot_se:.2f} points"
      f"   (formula said {record_restart['se_pts']:.2f})")

assert abs(rederived_gap - record_restart["estimate_pts"]) < 0.01, \
    "count-based rederivation disagrees with the pipeline — investigate"
assert abs(boot_se - record_restart["se_pts"]) < 0.15, \
    "bootstrap spread far from the formula uncertainty — investigate"
print("\n✓ Both quantities rederived. Two different methods, one answer each —")
print("  the computation is verified; the data and the design still are not.")

**Reading the output.** The counts give **+3.41**, agreeing with the pipeline
to the decimal, and the bootstrap spread lands near **1.21** against the
formula's **1.23**: two different uncertainty methods, close but not identical,
which is exactly how honest agreement between different machinery looks. Now
say precisely what you have earned. Agreement across methods verifies the
**computation**: the arithmetic from this table to that number is right. It
cannot verify the **inputs** or the **design**: if the file itself were wrong,
or a filter upstream changed the question, both methods would agree on the same
wrong answer, in perfect harmony.

> **A question that often comes up here:** *"My AI assistant already walked
> through the code line by line and confirmed it, and a second AI review agreed.
> Isn't that my independent check?"* Two readings are two opinions about the
> same code, and they can share the same blind spot; agreement between readers
> is not agreement between methods. The rederivation you just ran took a
> different computational path entirely. That is what "independent" buys, and
> it is why M7 asks for rederived numbers, not collected opinions.

**Section bridge:** one line of the colleague's dossier is still unexplained,
the forecast that improved on its own.

---

## 4. The Prediction-Route Branch: the Leakage and Boundary Audit

**Guiding question:** *a forecast's score jumped after a rerun, so what must
you rule out before you believe any of it?*

**Route label:** if your Contract declares the **prediction pathway**, this
branch is a required part of your M7 audit. Every other route: read it once,
as the sharpest claim-trace failure you will ever see, then apply its lesson
to your own claim language in section 5.

The colleague's note said the restart "picked up" a new feature,
`post_election_contact`, and the forecast jumped from 0.65 to 0.89. That
feature is a canvasser's log flag tallied *after* the election, from the same
rolls that record turnout. **Data leakage** is a feature carrying information
that would not exist at the moment of prediction, its value settled at or
after the outcome it claims to forecast. Example: predicting hospital
readmission from a discharge code recorded only when the patient leaves. The
test is never the score. The test is **timing**: is this feature's value
already settled at prediction time? A leak is also exactly the kind of change
a run-record diff catches, because the feature list of the fitted model
belongs in the record.

> **A question that often comes up here:** *"If the leak makes held-out
> accuracy go UP, why is it bad? Isn't higher better?"* Higher on this file,
> useless in the world. The leaky feature will not exist, or will not yet be
> settled, when a real forecast is needed, so its points are borrowed from the
> future. A 0.89 that read the answer key forecasts nothing; the honest 0.65
> is the number that survives contact with a case whose outcome is unknown.

**What to expect when you run the next cell:** the leak rebuilt on the voter
file: an honest baseline, a clean model, the leaky model's fake jump, and the
two checks that convict it.

### 🛠️ Run It Live: the leak, caught by timing

Run the cell below and watch the three accuracies in order: baseline, clean,
leaky.

**🏠 Optional depth.** Run the AI prompt on your own if you want to go deeper.
**Before you ask:** commit one sentence first: for YOUR project's data, which
single field is most likely to be settled after your outcome?

> 💡 **AI Prompt:** "Here is the outcome I forecast and the moment the forecast
> would be needed: [your outcome, and when]. Here are my candidate features,
> each with when its value is recorded: [your list]. For each feature, say
> whether its value is settled before, at, or after the outcome, and flag any
> that could not exist at prediction time. Do not use model accuracy to defend
> any feature; the timing decides."
>
> **After running, verify (counters *plausible-but-wrong-method*: a fluent case
> for keeping a high-scoring feature is still the wrong method for a forecast):**
> - [ ] Check the flag against the feature you committed above. If the tool
>   cleared a feature you know is settled late, the tool failed the timing
>   test, and your timeline wins.
> - [ ] Reject any defense of a feature that cites its accuracy contribution.
>   Timing convicts or clears; the score never does.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# THE PREDICTION-ROUTE AUDIT — baseline, clean model, leaky model, timing verdict.
voters = load_course_data("la_voter_file.csv")
assert voters.shape == (1000, 4), "unexpected shape — investigate before modeling"
print("✓ Loaded la_voter_file.csv —", voters.shape[0], "rows,", voters.shape[1], "columns")

y_v = voters["voted_2012"]
X_v = pd.get_dummies(voters[["party", "age"]], columns=["party"], drop_first=True).astype(float)
Xv_tr, Xv_te, yv_tr, yv_te = train_test_split(
    X_v, y_v, test_size=0.25, random_state=SEED, stratify=y_v)

base_acc = accuracy_score(
    yv_te, DummyClassifier(strategy="most_frequent").fit(Xv_tr, yv_tr).predict(Xv_te))
clean_acc = accuracy_score(
    yv_te, make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    .fit(Xv_tr, yv_tr).predict(Xv_te))

# The colleague's feature: a post-election log flag, i.e. the outcome with 15% noise.
leak_rng = np.random.default_rng(SEED)
flip = leak_rng.random(len(voters)) < 0.15
voters["post_election_contact"] = np.where(
    flip, 1 - voters["voted_2012"], voters["voted_2012"]).astype(float)
X_leaky = X_v.assign(post_election_contact=voters["post_election_contact"].values)
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    X_leaky, y_v, test_size=0.25, random_state=SEED, stratify=y_v)
leaky_acc = accuracy_score(
    yl_te, make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    .fit(Xl_tr, yl_tr).predict(Xl_te))

print(f"\n  baseline (always guess the majority):  {base_acc:.3f}")
print(f"  clean model (party + age):             {clean_acc:.3f}")
print(f"  leaky model (+ post_election_contact): {leaky_acc:.3f}   <- the colleague's 0.89")

# The two checks that convict the leak:
r = np.corrcoef(voters["post_election_contact"], voters["voted_2012"])[0, 1]
print(f"\n  CHECK 1 (correlation): corr with the outcome = {r:.3f} — nearly a copy")
print("  CHECK 2 (timing): tallied AFTER turnout is known — cannot exist at prediction time")
print("  VERDICT: drop the feature; the honest model is the clean one.")

fig, ax = plt.subplots(figsize=(7.5, 4.8))
bars = ax.bar(["Clean model\n(honest)", "Leaky model\n(reads the answer)"],
              [clean_acc, leaky_acc], color=["#4C72B0", "#D1651A"],
              edgecolor="white", hatch=["", "//"])
ax.axhline(base_acc, color="#8C8C8C", linestyle=":", linewidth=1.5,
           label=f"baseline = {base_acc:.3f}")
ax.set_ylim(0, 1)
ax.set_ylabel("Held-out accuracy")
ax.set_title("The jump is borrowed from the future: the hatched bar is not skill")
ax.legend(loc="lower right")
for b, v in zip(bars, [clean_acc, leaky_acc]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

assert leaky_acc > clean_acc + 0.15, "the leak should inflate the score sharply"
assert clean_acc > base_acc, "the clean model should still clear the baseline"
print(f"✓ Leak convicted: the feature bought {leaky_acc - clean_acc:+.1%} of fake accuracy.")

*The chart shows two bars against a dotted baseline: the clean model slightly
above it, and the hatched leaky bar towering over both. The hatching marks the
score that timing disqualifies.*

**Reading the output.** The baseline scores **0.604** for free, the clean model
earns **0.648**, and the leaky model fakes **0.892**. The correlation check
reads **0.709**, nearly a copy of the outcome, and the timing check ends the
argument: a flag tallied after the election cannot feed a forecast of it. So
the colleague's "good news" was a leak that a run-record diff would have caught
as a changed feature list. The claim language a prediction route may carry into
M7 is exactly this bounded: "on held-out cases the model never trained on, my
model beats the baseline by 4.4 points; I checked every feature's timing, and I
make no causal reading of the model's weights." A feature towering atop an
importance ranking explains nothing; importance is association inside one
model, and a leak manufactures it at will.

**Section bridge:** every route now owns a verdict; what remains is the
vocabulary to file it under, and the sentences it licenses.

---

## 5. Diagnose and Decide: the Discrepancy Taxonomy

**Guiding question:** *something differs between two runs, so what kind of
failure is it, and what may you still claim?*

When a clean restart disagrees with a record, the diagnosis lands in one of
five lanes. Learn the five words; your M7 discrepancy log uses them by name.

- **Data**: the input itself changed between runs. Example: the loader now
  returns 1,012 rows where the record says 1,000, and the fingerprint moved.
- **Environment**: the software changed around unchanged code. Example: your
  hosted runtime upgraded pandas overnight, and a default changed under you.
- **Order**: hidden state let the session's numbers detach from the shipped
  cell order. Example: the colleague's late cleaning cell.
- **Code**: the code itself changed, or was wrong all along. Example: a
  well-meaning edit to the transform cell after the record was saved.
- **Claim trace**: every number reproduces, and a written sentence still
  matches no output. Example: a write-up says "5 points" while every cell
  prints 3.41.

Two rules govern the verdict. First, **the difference can be the finding**: "my
result did not survive its clean restart, the lane was order, and here is the
repair" is a legitimate, gradable M7 outcome, reported exactly that way. Second,
**unresolved stays unresolved**: a discrepancy you cannot yet diagnose is
reported as open, and it blocks every stronger claim downstream until it
closes. What the verdict never does is crown a run by its timestamp.

> **A question that often comes up here:** *"Two of my numbers moved in the
> third decimal after a restart. Did I fail?"* Set a tolerance in advance and
> write it into your record; then the question answers itself. With recorded
> seeds, a fully deterministic pipeline reproduces to the last digit, so even a
> third-decimal drift is information: it usually means an unseeded step or an
> environment change, and it goes in the log with a lane word like everything
> else.

### 📝 Practice: classify the discrepancy, then audit the claim

*(In class: answers aloud, a couple of minutes; writing them down is optional.)*

*(Human-first: do both parts yourself before any AI. The sorting is the graded
skill.)*

**Part 1, one lane word each.** Five colleagues report five clean-restart
surprises. File each under data, environment, order, code, or claim trace.

- **A.** "The restart matched every number, but printed a warning that my
  hosted runtime upgraded pandas since my record was saved."
- **B.** "My loader pulled 1,012 rows today; my record says 1,000, and the
  fingerprint moved."
- **C.** "Every cell reproduces exactly, but my poster draft says 5 points and
  no output anywhere prints 5."
- **D.** "My number only comes back if I run the modeling cell before the
  cleaning cell, which is not the order they appear in."
- **E.** "My teammate tidied the transform cell yesterday; the diff shows one
  edited line, and the estimate moved."

**Part 2, licensed or not.** For each sentence about the shared Foos result,
say whether the clean-restart evidence licenses it, and if not, name the lane
or boundary it violates.

- **F.** "From a clean restart, the pipeline reproduces a +3.41-point gap with
  a 95% interval of +1.0 to +5.8."
- **G.** "The result reproduced, so the finding is correct."
- **H.** "The campaign raised turnout by 4 points."
- **I.** *(prediction routes)* "The model's top-ranked feature is what drives
  turnout."

### YOUR ANSWER HERE:

**Part 1 (A / B / C / D / E, one lane word each):**

**Part 2, F (licensed? why):**

**Part 2, G (licensed? why):**

**Part 2, H (licensed? why):**

**Part 2, I (licensed? why):**

---

### 🔁 Modify the Prompt

*(In class: one modification, one prediction, one run; deeper variations are optional depth.)*

The live prompt audited two run records the notebook handed you. Now point the
same move at your own package, and add the habit this week is named for:
**demand the execution record.** First, in one sentence, name the claim in your
own package you most doubt would survive a clean restart. Do not let AI pick it.

> **Base prompt (the run-record audit):** "Here are two run records from the
> same notebook: [paste]. For each field that differs, name which failure lane
> it points to (data, environment, order, code, claim trace) and the one check
> that would confirm the diagnosis. Do not declare either run correct."

Adapt it three ways: swap in your own submitted record and your own
clean-restart record; name your route, so the tool knows whether a feature-list
change matters; and append one standing instruction: "for every number you
report back to me, include its execution record: inputs, outputs, seed, and
the order things ran." In the cell below: paste your adapted prompt, predict
which field it will flag first, run it, and note whether the flag matched your
own diff, and whether every number it returned came with its record.

### YOUR ANSWER HERE:

**The claim of mine I most doubt survives a restart (named before asking):**

**My adapted run-record prompt (my records, my route, the standing instruction):**

**The field I predicted it would flag first:**

**What it flagged, whether that matches my diff, and whether its numbers carried records:**

---

### 🔬 Interrogate the Output

*(In class: raise the sharpest challenge and check it; the full written
interrogation is optional depth.)*

Suppose your AI assistant reads your notebook and returns a confident verdict:
**"Everything reproduces; your result is verified."** Do not accept it, and do
not reject it on reflex either. Interrogate it against your own evidence, using
four checks, and answer each in the cell below.

- **Claims:** verified against what? A verdict is only as good as the diff
  behind it. If no run-record comparison exists yet, the sentence described a
  hope, not a check.
- **Assumptions:** a reading of code is not a run of code. Ask what the verdict
  is based on: if the tool only read the cells, it is predicting what a clean
  restart would show, and your actual restart outranks its prediction.
- **Missing information:** which lanes did the verdict never mention? A
  verification that skips the claim trace, or never asks what changed in the
  environment, left out the checks that decide.
- **Overstatements:** "verified" covering the design. Even a perfect diff
  verifies the pipeline only; flag any word that quietly extends it to the
  claim.

And the rule that governs the whole week: **a clean run is not a correct
claim.** A notebook can restart beautifully and still feed a sentence no output
supports. Verify the number, its trace, and its boundary, never just the green
check.

### YOUR ANSWER HERE:

**Claims (what diff, if any, stands behind the verdict):**

**Assumptions (reading vs running — what was it actually based on):**

**Missing information (the lanes it never mentioned):**

**Overstatements (the exact words that claim too much):**

---

### 🧑‍⚖️ Human-Only Checkpoint

*(In class: AI closed, one decision, one line of reasoning.)*

Close your AI for this one. No AI, no lookup. The verdict on what your M7
reports is a never-delegate decision: it is your name on the record. In the
cell below, write, in your own words:

1. **Your pipeline's clean-restart status, honestly labeled:** survived, a
   number moved, or not yet run. All three are legitimate states to report
   today; only pretending is not.
2. **If a number moved (or when one does):** the lane you suspect, the one
   check that would confirm it, and your reporting decision, remembering that
   the difference can be the finding and that unresolved stays unresolved.
3. **The two quantities you will independently rederive for M7,** each with its
   second method named. Prediction routes: add the one feature whose timing
   you will check first.

If you cannot yet name a second method for your headline number, that is a
finding too: it tells you the number has exactly one path behind it, and your
verification plan starts there.

### YOUR ANSWER HERE:

**1. Clean-restart status (survived / a number moved / not yet run):**

**2. If moved: suspected lane + confirming check + reporting decision:**

**3. Two quantities and their second methods (+ the timing check, prediction routes):**

---

### ⚖️ Make a Design Choice: when the number moves, what ships?

*(In class: commit to one option in a single written line and be ready to
defend it aloud; the full write-up is optional depth.)*

*(Human-first: settle your own choice and its defense before you ask any AI.)*

Your clean restart returns a number that differs from the one you submitted.
The milestone is due at the end of the week. Choose the rule you would follow,
and commit it in one written line:

- **A.** Report the new number: the clean restart is the cleaner run, so its
  output supersedes the session's.
- **B.** Report the submitted number: it is the one on record, and the rerun
  can be footnoted.
- **C.** Report the diagnosis: neither number ships until the difference has a
  lane, the lane has a check, and the decision the check forces is recorded;
  if it cannot close in time, the discrepancy is reported as the finding.

### YOUR ANSWER HERE:

**My rule (A / B / C) and one line of defense:**

---

### 🎯 Take It to Your Project: the spine of your M7 verification record

In class, one sentence: name the first piece of your M7 verification record
you will produce, the clean-run log, the run-record diff, the rederivation
pair, or, on a prediction route, the timing audit, and why that piece first.
Write it below, then add the same line to your Research Project Dossier.

**The record you assemble at Friday's studio (M7, Clean-Restart Verified
Analysis):** your frozen first-analysis package rerun clean · the run-record
diff, with any discrepancy given its lane and either resolved or honestly
bounded · two independently rederived quantities · the updated claim-output
trace, where every sentence names the output that backs it · the
prediction-route timing/split/leakage appendix where it applies · your ledger
rows. The M7 brief collects all of it; your AI assistant helps there, and the
verdict stays yours.

### YOUR ANSWER HERE:

**My line (the M7 piece I produce first, and why that one):**

---

### 🛡️ Defend Your Decision

Defense #08 — the short ritual close, one line each:

1. **The claim I can defend:** one bounded sentence about your result's
   verification status, as it stands today.
2. **Its boundary:** what a clean restart verifies (the pipeline) and what it
   never verifies (the design, the claim's truth), one line.
3. **My uncertainty and limitations:** the part of your result that remains
   least independently checked, one line.
4. **AI check:** what you delegated this week, and the record or rederivation
   that decided what you kept.

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary:**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in the ledger. One worked row is filled in
as a model, and it captures the discipline this week teaches: **the diagnosis
was checked against the diff; the verdict was never outsourced.** This notebook
offers four prompts, one live in class and three at optional depth, so your
ledger carries a row for each one you actually ran, not a fixed count. The
ledger is a graded habit, not paperwork: it is how you show your work was
verified.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Audit the field-by-field diff of two run records | your AI | "Here are two run records from the same notebook: [pasted]. For each differing field, name the failure lane and the check that would confirm it. Do not declare either run correct." | Correct lane call on the row-count change; then drifted into crowning the newer run as "the corrected result" | Kept the lane diagnosis; rejected the crowning sentence, because neither run is right by order | Checked every flagged field against my printed diff; confirmed the lane with the reconstruction cell | The verdict sentence sounded like a conclusion and nearly slipped into my notes | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

## 6. Wrap-Up

In one lecture your first result grew up. You ran the shared pipeline from a
clean restart, watched every field of its run record match the submitted one,
and took the provisional label off, for the pipeline, in that environment. You
met the colleague whose number did not survive: a late cleaning cell, hidden
state, a submitted +3.41 that the shipped code no longer produces, and you
refused the reflex of crowning either run, because neither run is right by
order. You rederived the estimate from raw counts and the uncertainty by
bootstrap, and said precisely what that agreement verifies: the computation,
never the inputs or the design. The prediction routes convicted a leak by
timing rather than by score. And you filed every possible surprise into five
lanes, data, environment, order, code, claim trace, with the two governing
rules: the difference can be the finding, and unresolved stays unresolved.

> **"Neither run is right because of when it ran. A number that moves on a
> clean restart is a finding about your pipeline, so diagnose it, name its
> lane, and report the diagnosis. Demand the execution record, rederive the
> key numbers yourself, and let no claim outrun the output it traces to."**

Friday's studio develops **M7, your Clean-Restart Verified Analysis** (Book
Milestone 7, version 2): the clean-run log, the run-record diff with every
discrepancy resolved or honestly bounded, two independent rederivations, the
verified claim-output trace, and the prediction-route leakage appendix where
it applies. Bring your frozen first-analysis package exactly as you submitted
it. Next week the verified result meets its hostile reviewer: pre-listed
robustness checks, negative tests, and adversarial review, where a result that
survived its restart earns the right to be attacked properly. This notebook
companions **EDR|AI ch. 22, 'AI as Analytical Assistant,'** and revisits
**ch. 21, 'AI as Programmer.'**

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *EDR|AI ch. 22 'AI as Analytical Assistant' | demand the execution record, independent rederivation of key numbers, verify every claim against a specific output | adapted (course-lab version of the chapter's discipline)*
- *EDR|AI ch. 21 'AI as Programmer' (revisit) | the clean-run standard, the environment record, and the claim-output trace this lecture verifies | revisited*
- *foos_etal.csv | rdss package data | Foos et al. UK get-out-the-vote field experiment replication; unweighted difference in means (+3.41 points, se 1.23) rerun from a clean restart; the colleague reconstruction drops ward G to stage the order-lane failure (+4.27) | adapted (classroom-simple version)*
- *la_voter_file.csv | rdss package data | prediction-route leakage branch only: majority-class baseline, clean logistic model (party + age), a leaky post-election feature convicted by correlation and timing | adapted (shortened from the retired full prediction lab; the route's full craft lives in book ch. 16)*
- *course replication module | restart-and-run-all, the run-record comparison, and the execution-record prompt pattern, simplified for a first verification | adapted*
- *fresh | the run-to-run diff exercise, the five-lane discrepancy taxonomy (data / environment / order / code / claim trace), the independent-rederivation pair (crosstab counts + seeded bootstrap), the all-route claim-language audit, and the hidden-state teaching bug (seed 464) | newly-constructed-from-source-concept*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Moreira, D. *Evidence-Driven Research in the Age of AI* (EDR|AI), ch. 22
  'AI as Analytical Assistant' (required):
  [the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part4-credible-evidence/20-ai-as-analytical-assistant.html);
  revisit ch. 21 'AI as Programmer':
  [the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part4-credible-evidence/19-ai-as-programmer.html).
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences* (the dataset's companion book; recommended background). Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>

---

### ⏸ Optional depth from here

**Today's lecture path and the notebook's close are complete.** Everything below this line is optional depth: run it if you want to push the ideas further. Nothing here is required, and any 🏠-marked prompt above is optional too.

## 8. Optional Depth: Break a Pipeline on Purpose

**Guiding question:** *what does hidden state feel like from the inside, and
what is the smallest habit that makes it impossible?*

The colleague's bug is easier to avoid once you have built it yourself. The
cell below stages the whole failure in miniature: an estimate, a cleaning step
that **mutates** the shared table (changes it in place, so every later cell
sees the altered version under the same name), and the two different numbers
the two run orders produce. It then shows the repair: derive each downstream
table into its **own name**, so every number states which input it came from
and no cell can be silently re-fed. It also recomputes the data fingerprint on
both tables, so you can watch the data lane flag what the order lane caused.

**🏠 Optional depth.** Run the AI prompt on your own if you want to go deeper.
**Before you ask:** in one sentence, predict which of the five lanes a reviewer
who only saw the two fingerprints would name first, and why that diagnosis
would be reasonable but incomplete.

> 💡 **AI Prompt:** "This notebook fragment computes an estimate, then a
> cleaning step reassigns the same table name, and the same estimate cell gives
> a different number depending on run order: [paste the next cell]. Explain the
> mechanism, then propose the smallest rewrite that makes every downstream
> number independent of run order, and say which discrepancy lane each version
> fails or passes under a clean restart."
>
> **After running, verify (counters *plausible-but-wrong-method*: a fluent fix
> can add machinery without removing the order dependence):**
> - [ ] Apply its rewrite mentally to the cell: if any downstream number still
>   depends on which cell ran last, the fix failed regardless of how tidy it
>   reads.
> - [ ] Confirm its lane assignments against your own: the mutating version
>   fails order; the renamed version passes it, and both leave the design
>   question (drop ward G or not) untouched.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# OPTIONAL DEPTH — build the hidden-state bug, feel it, then make it impossible.
def fingerprint(df):
    return int(pd.util.hash_pandas_object(df).sum()) % 10**12

# THE FRAGILE VERSION: one shared name, reassigned in place.
table = foos.copy()
gap_before, n_before = estimate_gap(table)      # estimate runs first...
fp_before = fingerprint(table)
table = table[table["ward"] != "G"]             # ...then the cleaning cell REASSIGNS it
gap_after, n_after = estimate_gap(table)        # same call, different world
fp_after = fingerprint(table)

print("FRAGILE (one shared name):")
print(f"  estimate before cleaning: {gap_before:+.2f} points on {n_before:,} rows  (fingerprint {fp_before})")
print(f"  estimate after  cleaning: {gap_after:+.2f} points on {n_after:,} rows  (fingerprint {fp_after})")
print("  Same cell, two answers — which one you get depends on run order alone.")

# THE ROBUST VERSION: every derived table gets its own name; each number names its input.
full_table = foos.copy()
no_ward_g = full_table[full_table["ward"] != "G"]
gap_full, n_full = estimate_gap(full_table)
gap_cut, n_cut = estimate_gap(no_ward_g)

print("\nROBUST (one name per table):")
print(f"  gap on full_table: {gap_full:+.2f} points on {n_full:,} rows")
print(f"  gap on no_ward_g:  {gap_cut:+.2f} points on {n_cut:,} rows")
print("  Both numbers now exist, each labeled with its input — run order cannot swap them.")
print("  Which one the project reports is a design decision for the Contract, not the kernel.")

assert abs(gap_full - 3.41) < 0.05 and abs(gap_cut - 4.27) < 0.05
assert fp_before != fp_after, "the fingerprint should expose the reassigned table"
print("\n✓ Hidden state built, felt, and retired. The habit: derive into new names,")
print("  never overwrite a shared table, and let every printed number name its input.")

*In plain terms: the fragile version gives two answers from one cell depending
on run order, while the robust version gives two labeled answers that no run
order can confuse.*

The later replication module turns this same discipline outward: there you
cold-run a stranger's package and log every place its numbers depend on
anything but its files. Every habit from today, the record, the diff, the
lanes, the rederivation, is what makes your own package survive that
treatment.